# VoxTool Real-Adapter Colab Demo

This notebook demonstrates the advanced benchmark adapters (Voxtral, Qwen,
Gemma) behind the shared `ModelAdapter` interface. It selects an adapter
config, runs text examples through Pipeline A, validates the canonical JSON
envelope, executes tools through the registry, shows final answers, and
displays metrics. The mock adapter requires no download; real adapters fetch
weights lazily on first use and need GPU/Colab resources.

Ordinary CI never runs this notebook against real models. Run real adapters
manually here in Colab.

## 1. Install dependencies

Install the project plus the optional model extras. In Colab this pulls the
heavy `transformers`/`torch` stack used only for real adapters.

In [ ]:
# Colab dependency install (skip when running a local checkout).
%pip install -q -e '.[model,notebook]'  # noqa

## 2. Load the repository

Clone the repo in Colab and add it to `sys.path` so the packages import.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/nkz-soft/voxtool.git"
if not os.path.exists("voxtool"):
    !git clone -q $REPO_URL  # noqa
sys.path.insert(0, os.path.abspath("voxtool"))

## 3. Select an adapter

List the registered adapters and build one from its config. No model is
downloaded at this step.

In [ ]:
from apps.notebook import colab_demo_helpers as demo

print("available adapters:", demo.list_adapters())
# Use 'mock' for a no-download dry run, or 'qwen'/'gemma'/'voxtral' for real.
ADAPTER_ID = "mock"
adapter = demo.select_adapter(ADAPTER_ID)
adapter.adapter_id, adapter.capabilities.model_dump()

## 4. Run text examples

Run a few prompts through Pipeline A. Each record preserves raw output,
parsed JSON, validation results, tool execution, and the final answer.

In [ ]:
prompts = [
    "Convert 2 kilometers to meters.",
    "What is the capital of France?",
]
records = demo.run_text_demo(adapter, prompts, run_id="colab-demo")
[demo.record_summary(r) for r in records]

## 5. Optional: upload audio

Audio pipelines (C/D) require an adapter with audio capabilities such as
Voxtral. Upload a short clip in Colab; this cell is optional and skipped for
text-only adapters.

In [ ]:
# Optional audio upload (Colab only).
try:
    from google.colab import files  # type: ignore

    if "supports_audio_input" and adapter.capabilities.supports_audio_input:
        uploaded = files.upload()
        print("uploaded:", list(uploaded))
    else:
        print("Selected adapter has no audio input; skipping upload.")
except ImportError:
    print("Not running in Colab; audio upload skipped.")

## 6. Validation, execution, and final answers

Inspect parse status, validation errors, executed tool results, and final
answers for each example.

In [ ]:
import pandas as pd

summary = pd.DataFrame(demo.record_summary(r) for r in records)
summary[["example_id", "parsable", "validation_errors", "tool", "final_answer"]]

## 7. Metrics

Show simple aggregate metrics over the demo run.

In [ ]:
total = len(records)
parsable = sum(1 for r in records if r.first_pass_parsable)
with_tool = sum(1 for r in records if r.tool_execution_result is not None)
{
    "examples": total,
    "parsable_rate": parsable / total if total else 0.0,
    "tool_calls": with_tool,
}